# Lesson 3.1 — experiments: games and minimax

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.games import reference as algorithms

The function `run` runs one search and returns its result, the number of positions visited and the time in seconds. Positions are counted by the calls of `game.result`, and the start position counts as visited too. The function `positions_by_ply` walks the whole game tree and counts the nodes and the distinct positions at every level. Minimax with a transposition table (`memoized_minimax_search`) comes from the course's reference version.

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt

from introai.games import TIC_TAC_TOE, reference
from introai.games.players import play, random_player
from introai.testing.games import positions_visited
from introai.trace import TraceLimitExceeded, record

HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'


def run(search, game, state, limit=1_000_000):
    """Run one search; return its result, the positions visited and the time in seconds."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"result"}) as trace:
            result = search(game, state)
    except NotImplementedError:
        raise RuntimeError(f"{search.__name__} is not written yet - {HINT}") from None
    except (TraceLimitExceeded, RecursionError):
        message = f"{search.__name__} visited more than {limit} positions - {HINT}"
        raise RuntimeError(message) from None
    return result, positions_visited(trace), time.perf_counter() - start


def positions_by_ply(game):
    """The plies, the nodes of the game tree and the distinct positions at every ply."""
    nodes, distinct = Counter(), {}
    stack = [(game.initial, 0)]
    while stack:
        state, ply = stack.pop()
        nodes[ply] += 1
        distinct.setdefault(ply, set()).add(state)
        if not game.is_terminal(state):
            stack.extend((game.result(state, a), ply + 1) for a in game.actions(state))
    plies = sorted(nodes)
    return plies, [nodes[p] for p in plies], [len(distinct[p]) for p in plies]

## Experiment 1: the whole tic-tac-toe tree

Looking for the first move on the empty board, minimax walks the whole game tree. We compare it with a minimax that remembers the positions already solved. The computation takes a few seconds.

**Your prediction.** How many positions will your minimax visit? How many of them are distinct? How many times less work will the version with a memory do?

*My prediction:* …

In [ ]:
(value, move), visited, seconds = run(algorithms.minimax_search, TIC_TAC_TOE, TIC_TAC_TOE.initial)
_, memo_visited, memo_seconds = run(
    reference.memoized_minimax_search, TIC_TAC_TOE, TIC_TAC_TOE.initial
)
plies, nodes, distinct = positions_by_ply(TIC_TAC_TOE)
print(f"value of the empty board: {value}, first move: {move}")
print()
print(f"{'':<24}{'positions':>10}{'seconds':>9}")
print(f"{'minimax':<24}{visited:>10}{seconds:>9.2f}")
print(f"{'minimax with a table':<24}{memo_visited:>10}{memo_seconds:>9.2f}")
print(f"{'distinct positions':<24}{sum(distinct):>10}")
print()
repeats = sum(nodes) / sum(distinct)
print(f"the tree has {sum(nodes)} nodes: a position appears {repeats:.0f} times on average")

**Think about it.** The version with a memory also calls `game.result` more often than there are distinct positions. Why? In which lesson of module 2 did the same idea appear, and what was it called there?

## Experiment 2: how many nodes there are at each level

At every level of the game tree, from the empty board (level 0) to the full one (level 9), we count the nodes and the distinct positions. The vertical axis has a logarithmic scale.

**Your prediction.** At which level does the tree have the most nodes? Why not at the last one?

*My prediction:* …

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
width = 0.4
ax.bar(
    [p - width / 2 for p in plies],
    nodes,
    width,
    color="#2a78d6",
    edgecolor="#fcfcfb",
    linewidth=1,
    label="nodes of the game tree",
)
ax.bar(
    [p + width / 2 for p in plies],
    distinct,
    width,
    color="#eb6834",
    edgecolor="#fcfcfb",
    linewidth=1,
    label="distinct positions",
)
peak = max(plies, key=lambda p: nodes[p])
ax.annotate(
    f"{nodes[peak]:,}",
    (peak - width / 2, nodes[peak]),
    textcoords="offset points",
    xytext=(0, 4),
    ha="center",
    color="#0b0b0b",
)
ax.set_yscale("log")
ax.set_xticks(plies)
ax.set_title(
    "Tic-tac-toe: nodes and distinct positions at each ply (log scale)",
    loc="left",
    color="#0b0b0b",
)
ax.set_xlabel("ply: moves made so far", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Think about it.** From the fifth ply on, some games are already over. How does the chart show it? At which level is the gap between the number of nodes and the number of distinct positions largest, and where does it come from?

## Experiment 3: games

First minimax plays against itself. Then it plays 100 games against a player who picks moves at random. In even-numbered games minimax plays crosses, in odd-numbered ones noughts. Minimax is deterministic, so the player `remembering` remembers its move in every position and does not compute it twice.

**Your prediction.** How will the game of minimax against minimax end? How many games against the random player will minimax lose? Will it win them all?

*My prediction:* …

In [ ]:
def remembering(search):
    """A player that plays the move of `search` and remembers it in every position."""
    moves = {}

    def player(game, state):
        if state not in moves:
            (_, moves[state]), _, _ = run(search, game, state)
        return moves[state]

    return player


perfect = remembering(algorithms.minimax_search)
duel = play(TIC_TAC_TOE, {"X": perfect, "O": perfect})
print(f"minimax against minimax: {duel.winner or 'draw'} after {len(duel.moves)} moves")
print(TIC_TAC_TOE.render(duel.state))

results = {"X": Counter(), "O": Counter()}
for number in range(100):
    side, other = ("X", "O") if number % 2 == 0 else ("O", "X")
    result = play(TIC_TAC_TOE, {side: perfect, other: random_player(number)})
    if result.winner is None:
        results[side]["draw"] += 1
    else:
        results[side]["win" if result.winner == side else "loss"] += 1
print()
print(f"{'minimax plays':<15}{'wins':>6}{'draws':>7}{'losses':>8}")
for side in ("X", "O"):
    r = results[side]
    print(f"{side:<15}{r['win']:>6}{r['draw']:>7}{r['loss']:>8}")

**Think about it.** Did minimax lose a single game? Why does it win less often with noughts than with crosses? Did it win every game that could be won? Think about what minimax assumes about its opponent.